# YouTube Video RAG using LangChain & Gemini

This project implements a Retrieval-Augmented Generation (RAG) pipeline to query and analyze YouTube video transcripts. It automatically extracts captions using the YouTube Transcript API, chunks the text, and indexes it into a FAISS vector store using Google Gemini embeddings (`gemini-embedding-001`). When queried, the system retrieves the most relevant context and generates grounded answers using Google's Gemini LLM via LangChain Expression Language (LCEL).

In [1]:
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/var/folders/q0/c07cmrnn2rdfh6y463cwpklw0000gn/T/ipykernel_61935/3103007711.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Step 1: Document Ingestion (Transcript Extraction)
Fetches raw subtitle text from a specified YouTube video ID using the youtube-transcript-api and concatenates the transcript blocks into a single string.

In [23]:
video_id = "tCMzkJxBM5c"

try:
    fetched_transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en'])
    transcript_list = fetched_transcript.to_raw_data()
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")


Larry Fink, the guy running BlackRock, >> [music] >> is really the president of the United States if we look at the kind of influence he's got. Wait a minute. These guys essentially have a monopoly. Well, how big is BlackRock? >> Ladies and gentlemen, BlackRock is the most powerful, and according to US politicians, the most evil company in the world. >> [music] >> BlackRock, the largest asset manager in the world. >> [music] >> He can get a meeting with any company on the planet. >> [music] >> Jensen Huang built Nvidia from scratch, and today he owns 3.7% of Nvidia. But BlackRock, BlackRock owns 8.02%. Steve Jobs owns less than 1% of Apple, but BlackRock owns 7.97% of Apple. Bill Gates owns 1.34% of Microsoft, but BlackRock owns 8.18% of Microsoft. And just like this, the big three, which includes BlackRock, State Street, and Vanguard, they own a huge stake in almost every S&P 500 company. And BlackRock alone controls over 15 trillion dollars in assets. Do you realize how much 15 trill

In [3]:
len(transcript_list)

574

Step 2: Text Splitting
Breaks the long transcript string down into manageable, overlapping chunks using LangChain's RecursiveCharacterTextSplitter to maintain contextual flow across boundaries.

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.create_documents([transcript])

In [5]:
len(chunks)

51

Step 3: Embedding & Vector Store Indexing
Generates vector embeddings for each text chunk using Google's gemini-embedding model and stores them in a local FAISS vector database for efficient similarity search.

In [6]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    streaming=True,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model = 'gemini-embedding-2'
)


In [7]:
# Initialize the vector store
vector_store = FAISS.from_documents(chunks, embeddings)

Step 4: Retriever Setup
Configures a similarity search retriever over the FAISS vector store to fetch the top $k$ most relevant context chunks for any given user query.

In [8]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 4})
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10ecb16d0>, search_kwargs={'k': 4})

In [9]:
retriever.invoke("How does the blackrock works")


[Document(id='9240cc59-43dd-454b-b421-0f715da1c211', metadata={}, page_content="forth. This is the magic of ETFs that got them enormous stake in the Fortune 500 companies. So, now the question over here is the story so far seems pretty good. A failed banker found a gap in the market, built a trillion-dollar company, became a billionaire, end of story. [music] Then the question is, why is BlackRock being called evil, and why are the American politicians so [music] concerned about BlackRock? Well, as it turns out, BlackRock isn't alone. There are two more companies called"),
 Document(id='d987a92a-5e01-4941-986b-1a8056680add', metadata={}, page_content='trillion with a 0.03% commission running on autopilot. That is how BlackRock became a money-making machine. And every dollar invested in their S&P 500 bought them a stake in Nvidia, Apple, Microsoft, and 497 [music] of the most powerful companies on Earth. That is how BlackRock today owns 8.18% of Microsoft, 8.02% of Nvidia, 6.69% [music]

Step 5: Augmentation & Prompt Template
Defines a custom PromptTemplate that injects the retrieved transcript context alongside the user's question, instructing the Gemini model to answer strictly based on the provided text.

In [10]:

prompt= PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say get the f*ck out of here, I don't know.

    {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [11]:
question       = "How big is blackrock & how much assets does it holds?"
retrieved_docs= retriever.invoke(question)

In [12]:
retrieved_docs

[Document(id='4fa449a8-e4ad-496b-93c0-072b4a2e2775', metadata={}, page_content="Larry Fink, the guy running BlackRock, >> [music] >> is really the president of the United States if we look at the kind of influence he's got. Wait a minute. These guys essentially have a monopoly. Well, how big is BlackRock? >> Ladies and gentlemen, BlackRock is the most powerful, and according to US politicians, the most evil company in the world. >> [music] >> BlackRock, the largest asset manager in the world. >> [music] >> He can get a meeting with any company on the planet. >> [music] >>"),
 Document(id='3adf50c4-e30e-4030-93a7-e2b7079ba7be', metadata={}, page_content="BlackRock, State Street, and Vanguard, they own a huge stake in almost every S&P 500 company. And BlackRock alone controls over 15 trillion dollars in assets. Do you realize how much 15 trillion dollars is? >> [music] >> It is more than the GDP of India, Japan, and France combined. And BlackRock is a company that never made a chip. They

In [13]:
content_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

final_prompt = prompt.invoke({"context": content_text, "question" : question})


In [14]:
answer= llm.invoke(final_prompt)
print(answer.content)

[{'type': 'text', 'text': 'Based on the provided transcript, BlackRock is the largest asset manager in the world (and described as the most powerful and evil company). It alone controls over 15 trillion dollars in assets, which is more than the GDP of India, Japan, and France combined.', 'index': 0, 'extras': {'signature': 'EmAKXgFpFH0TNlGMIFJ1acWhSwfAs3+CsfnZXoaER3lhU/PqzTtVS8YwIN4RrNXprZZEpRA29+6SglrWILlcfm2LqwuZ20yfNgfGu2QFOZAry5C1Ym2Iph+0ocq9bDSE5ao='}}]


Step 6: LCEL Chain Execution
Constructs a complete RAG pipeline using LangChain Expression Language (LCEL) via RunnableParallel and StrOutputParser, piping inputs from the retriever into the prompt, the Gemini LLM (chat-google-genai), and finally parsing the text output.

In [15]:

from langchain_core.runnables import RunnablePassthrough, RunnableLambda,RunnableParallel
from langchain_core.output_parsers import StrOutputParser
     

def format_docs(retrieved_docs):
  content_text="\n\n".join(doc.page_content for doc in retrieved_docs)
  return content_text

In [16]:

paraller_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [17]:
parser= StrOutputParser()
#know we will be making main chain having 1st two paraller chain then to next chain of  prompt-> LLM --> Parser
main_chain = paraller_chain | prompt | llm | parser

In [18]:
main_chain.invoke("What is Blackrock & Who is Larry?")


"Based on the provided transcript:\n\n* **BlackRock** is the largest asset manager in the world (also described as the most powerful and, according to US politicians, the most evil company in the world). It controls over 15 trillion dollars in assets, owns a huge stake in almost every S&P 500 company (and is the largest shareholder in hundreds of America's biggest companies alongside Vanguard and State Street), but it has never made a physical product, chip, phone, or owned a factory. \n* **Larry Fink** is the guy running BlackRock."